## Import libraries

In [ ]:
import kaleido #required
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import time
from plotly.subplots import make_subplots
import os
from sklearn.metrics import confusion_matrix, accuracy_score

## Import dataset and predicted results dataset

In [ ]:
rep_path='../../../'

# Test error results plots

In [ ]:
test_error_dataset = pd.read_excel(os.path.join(rep_path,'Data/Results/Force_Map_Estimation_Results/Cov_4/test_Error_Cov4_rev3.xlsx'), header=0)
test_error_dataset[[12, 13, 14]] = (test_error_dataset.iloc[:, 2:5].sub(test_error_dataset.iloc[:, 8:11].values)).abs()
test_error_dataset_force_filtered = test_error_dataset[test_error_dataset.iloc[:, 5] == 1]  
test_error_dataset_force_filtered.iloc[:, 4] *= -1
test_error_dataset_force_filtered = test_error_dataset_force_filtered[test_error_dataset_force_filtered.iloc[:, 4] <= 2]  

In [ ]:
# This code filters out transient sensor noise by deleting any force readings that do not appear in a stable, consecutive streak of at least three identical values.
test_error_dataset_force_filtered_sorted = test_error_dataset_force_filtered.sort_values(
    by=[test_error_dataset_force_filtered.columns[0], test_error_dataset_force_filtered.columns[1]],
    ascending=[False, False]
).reset_index(drop=True)

# Initialize an index array to mark rows for deletion
rows_to_delete = np.zeros(len(test_error_dataset_force_filtered_sorted), dtype=bool)

# Loop through the DataFrame to check for consecutive equal values
for i in range(len(test_error_dataset_force_filtered_sorted) - 2):
    # Check for 3 consecutive equal values
    if not (
        test_error_dataset_force_filtered_sorted.iloc[i, 0] == test_error_dataset_force_filtered_sorted.iloc[i + 1, 0] == test_error_dataset_force_filtered_sorted.iloc[i + 2, 0]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

    # Check for 3 consecutive equal values
    if not (
        test_error_dataset_force_filtered_sorted.iloc[i, 1] == test_error_dataset_force_filtered_sorted.iloc[i + 1, 1] == test_error_dataset_force_filtered_sorted.iloc[i + 2, 1]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

# Delete the marked rows
test_error_dataset_force_filtered_cleaned = test_error_dataset_force_filtered_sorted[~rows_to_delete]

In [ ]:
# Optionally, display the cleaned DataFrame
print("Size of test_error_dataset_force_filtered:", test_error_dataset_force_filtered.shape)
print("Size of test_error_dataset_force_filtered:", test_error_dataset_force_filtered_cleaned.shape)

Touch binary classification accuracy result

In [ ]:
# This code filters out transient sensor noise by deleting any force readings that do not appear in a stable, consecutive streak of at least three identical values.
test_error_dataset_sorted = test_error_dataset.sort_values(
    by=[test_error_dataset.columns[0], test_error_dataset.columns[1]],
    ascending=[False, False]
).reset_index(drop=True)

# Initialize an index array to mark rows for deletion
rows_to_delete = np.zeros(len(test_error_dataset_sorted), dtype=bool)

# Loop through the DataFrame to check for consecutive equal values
for i in range(len(test_error_dataset_sorted) - 2):
    # Check for 3 consecutive equal values
    if not (
        test_error_dataset_sorted.iloc[i, 0] == test_error_dataset_sorted.iloc[i + 1, 0] == test_error_dataset_sorted.iloc[i + 2, 0]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

    # Check for 3 consecutive equal values
    if not (
        test_error_dataset_sorted.iloc[i, 1] == test_error_dataset_sorted.iloc[i + 1, 1] == test_error_dataset_sorted.iloc[i + 2, 1]
    ):
        rows_to_delete[i:i + 3] = True  # Mark the rows for deletion

# Delete the marked rows
test_error_dataset_sorted_cleaned = test_error_dataset_sorted[~rows_to_delete]

# Extract the target and predicted classification output
y_true = test_error_dataset_sorted_cleaned.iloc[:, 5].values  # True target values 
y_pred = test_error_dataset_sorted_cleaned.iloc[:, 11].values  # Predicted output values

# Threshold the predicted values at 0.5
y_pred_binary = (y_pred >= 0.5).astype(int)

# Generate the confusion matrix using sklearn
cm = confusion_matrix(y_true, y_pred_binary)
print("Confusion Matrix:")
print(cm)

# Calculate classification accuracy
accuracy = accuracy_score(y_true, y_pred_binary)
print(f"Classification Accuracy: {accuracy:.4f}")

In [ ]:
# Calculate mean of x, y, force errors
mean_x_error = test_error_dataset_force_filtered_cleaned.iloc[:, 12].mean()
mean_y_error = test_error_dataset_force_filtered_cleaned.iloc[:, 13].mean()
mean_force_error = test_error_dataset_force_filtered_cleaned.iloc[:, 14].mean()

print("Mean of X error:", mean_x_error)
print("Mean of Y error:", mean_y_error)
print("Mean of Force error:", mean_force_error)

In [ ]:
# Define the ranges
ranges = [0.02, 0.09, 0.5, 1, 1.5, 2]

# Create labels for the ranges
labels = ['[0.02, 0.09)', '[0.09, 0.5)', '[0.5, 1)', '[1, 1.5)', '[1.5, 2)']

In [ ]:
# --- PRE-CALCULATE INDIVIDUAL EUCLIDEAN DISTANCES ---
# Using your existing abs error columns (8 and 9)
test_error_dataset_force_filtered_cleaned['euclidean_dist'] = np.sqrt(
    test_error_dataset_force_filtered_cleaned.iloc[:, 12]**2 + 
    test_error_dataset_force_filtered_cleaned.iloc[:, 13]**2
)

# Results for the test dataset

In [ ]:
# Clean up by removing the data from outside the edges:
# This keeps the internal 23x23 grid (Indices 1 through 23)

sor = (
    # Remove pixels whose centers are outside the 17.25mm internal span
    (test_error_dataset_force_filtered_cleaned.iloc[:, 2] > -0.375) |   # Removes 0.0
    (test_error_dataset_force_filtered_cleaned.iloc[:, 2] < -17.625) | # Removes -18.0
    (test_error_dataset_force_filtered_cleaned.iloc[:, 3] > -0.375) |   # Removes 0.0
    (test_error_dataset_force_filtered_cleaned.iloc[:, 3] < -17.625)   # Removes -18.0
)

filtered_indices_edge_sensor = sor

# Regions in-between the taxels (the gaps where AI interpolates)
filtered_indices_inbetween_taxels = (
    (~filtered_indices_edge_sensor) & 
    (
        # X-direction gaps
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 2] < -3.75) & (test_error_dataset_force_filtered_cleaned.iloc[:, 2] > -5.25)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 2] < -8.25) & (test_error_dataset_force_filtered_cleaned.iloc[:, 2] > -9.75)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 2] < -12.75) & (test_error_dataset_force_filtered_cleaned.iloc[:, 2] > -14.25)) |
        # Y-direction gaps
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 3] < -3.75) & (test_error_dataset_force_filtered_cleaned.iloc[:, 3] > -5.25)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 3] < -8.25) & (test_error_dataset_force_filtered_cleaned.iloc[:, 3] > -9.75)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 3] < -12.75) & (test_error_dataset_force_filtered_cleaned.iloc[:, 3] > -14.25))
    )
)

# Regions directly over the 4x4 taxel electrodes
filtered_indices_inside_taxels = (
    (~filtered_indices_edge_sensor) & 
    (~filtered_indices_inbetween_taxels) & 
    (
        # X-bounds for Taxel centers (consistent with 0.75 steps)
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 2] <= -1.5) & (test_error_dataset_force_filtered_cleaned.iloc[:, 2] >= -3.0)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 2] <= -6.0) & (test_error_dataset_force_filtered_cleaned.iloc[:, 2] >= -7.5)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 2] <= -10.5) & (test_error_dataset_force_filtered_cleaned.iloc[:, 2] >= -12.0)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 2] <= -15.0) & (test_error_dataset_force_filtered_cleaned.iloc[:, 2] >= -16.5))
    ) & (
        # Y-bounds for Taxel centers
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 3] <= -1.5) & (test_error_dataset_force_filtered_cleaned.iloc[:, 3] >= -3.0)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 3] <= -6.0) & (test_error_dataset_force_filtered_cleaned.iloc[:, 3] >= -7.5)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 3] <= -10.5) & (test_error_dataset_force_filtered_cleaned.iloc[:, 3] >= -12.0)) | 
        ((test_error_dataset_force_filtered_cleaned.iloc[:, 3] <= -15.0) & (test_error_dataset_force_filtered_cleaned.iloc[:, 3] >= -16.5))
    )
)

# Remaining areas (transitions/edges of electrodes)
filtered_indices_edge_taxels = (
    (~filtered_indices_edge_sensor) & 
    (~filtered_indices_inbetween_taxels) & 
    (~filtered_indices_inside_taxels)
)

filtered_indices_inside_edge_sensor = (~filtered_indices_edge_sensor)
filtered_indices_sensor_plus_outside = np.ones(len(test_error_dataset_force_filtered_cleaned), dtype=bool)

In [ ]:
# --- CALCULATE METRICS PER REGION ---

def get_region_metrics(df, indices, name):
    subset = df.loc[indices]
    if subset.empty:
        return None
    
    abs_x = subset.iloc[:, 12]
    abs_y = subset.iloc[:, 13]
    abs_f = subset.iloc[:, 14]
    dist  = subset['euclidean_dist']
    
    return {
        "name": name,
        "MEE": dist.mean(),
        "MAE_X": abs_x.mean(),
        "MAE_Y": abs_y.mean(),
        "MAE_Force": abs_f.mean(),
        "Spatial_Bias": np.sqrt(abs_x.mean()**2 + abs_y.mean()**2)
    }

In [ ]:
# Compute for each group
regions_data = [
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_inside_taxels, "Inside Taxels"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_inbetween_taxels, "In-between Taxels"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_edge_taxels, "Edge Taxels"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_inside_edge_sensor, "Sensor"),
    get_region_metrics(test_error_dataset_force_filtered_cleaned, filtered_indices_sensor_plus_outside, "Sensor plus outside edge")
]

# Print Results
for res in regions_data:
    if res:
        print(f"--- {res['name']} ---")
        print(f"MEE: {res['MEE']:.4f} mm | L2 Norm of MAE: {res['Spatial_Bias']:.4f} mm")
        print(f"MAE in X-direction: {res['MAE_X']:.4f} mm | MAE in Y-direction: {res['MAE_Y']:.4f} mm")
        print(f"MAE Force: {res['MAE_Force']:.4f} N\n")

In [ ]:
# --- UPDATE PLOT DATA ---
regions = [r['name'] for r in regions_data]
mean_mee_errors = [r['MEE'] for r in regions_data]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=regions,
    y=mean_mee_errors,
    name='Mean Euclidean Error',
    text=[f'{v:.2f}' for v in mean_mee_errors],
    textposition='auto',
    width=0.4,
    marker=dict(color='purple', opacity=0.7)
))

fig.update_layout(
    title='Mean Euclidean Error (MEE) by Region',
    xaxis_title='Regions',
    yaxis_title='MEE [mm]',
    template='plotly_white'
)
fig.show()

In [ ]:
# --- 1. Prepare Data for Force Plotting ---
# Extracting names and MAE Force values from our regions_data list
# This includes the "Sensor plus outside edge" category you requested
plot_regions = [res['name'] for res in regions_data if res is not None]
mae_force_values = [res['MAE_Force'] for res in regions_data if res is not None]

# --- 2. Create the Force Bar Chart ---
fig_force = go.Figure()

fig_force.add_trace(go.Bar(
    x=plot_regions,
    y=mae_force_values,
    name='Force MAE',
    text=[f'{val:.2f}' for val in mae_force_values],
    textposition='auto',
    width=0.4,
    marker=dict(color='green', opacity=0.6) 
))

# Update layout
fig_force.update_layout(
    title='Force Prediction Error (MAE) for Different Regions',
    xaxis_title='Regions',
    yaxis_title='Mean Absolute Error [N]',
    template='plotly_white'
)

# Display the plot
fig_force.show()

# --- 3. Print Results ---
# We pull these directly from regions_data to ensure mathematical consistency
print("FORCE ERROR ANALYSIS\n" + "="*30)

# Helper to find specific region by name in our list
def print_stat(name):
    match = next((item for item in regions_data if item["name"] == name), None)
    if match:
        print(f"Total Force Error for {name}: {match['MAE_Force']:.4f} N")

print_stat("Edge Taxels") # You previously called this 'Edge Sensor Region' in print
print_stat("In-between Taxels")
print_stat("Edge Taxels")
print_stat("Inside Taxels")
print_stat("Sensor")
print_stat("Sensor plus outside edge")

## Plot violin plot for test dataset

### Violin location

In [ ]:
# data_set_locations="sensor_plus_edges"
data_set_locations="sensor_area"

# Define the ranges
ranges = [0.02, 0.5, 1, 1.5, 2]

# Create labels for the ranges
labels = ['[0.02, 0.5)', '[0.5, 1)', '[1, 1.5)', '[1.5, 2]']

if data_set_locations=="sensor_plus_edges":
    filtered_test_dataset = test_error_dataset_force_filtered_cleaned[:] 
elif data_set_locations=="sensor_area":
    filtered_test_dataset = test_error_dataset_force_filtered_cleaned[filtered_indices_inside_edge_sensor]

# Create a new column 'category' based on the ranges
filtered_test_dataset['category'] = pd.cut(filtered_test_dataset.iloc[:, 4], bins=ranges, labels=labels, right=False)

# Specify x values
x = filtered_test_dataset['category']  # Categorical variable created based on the ranges

y_9th_column = filtered_test_dataset.iloc[:, 9]-filtered_test_dataset.iloc[:, 3]   
y_10th_column = filtered_test_dataset.iloc[:, 8]-filtered_test_dataset.iloc[:, 2]

# Calculate the square root of the sum of squares of the two columns
y_sqrt = np.sqrt(y_9th_column ** 2 + y_10th_column ** 2)

# Filter out negative values for y_sqrt
y_sqrt = y_sqrt[y_sqrt >= 0]

fig = go.Figure()

fig.add_trace(go.Violin(x=x, y=y_9th_column, points=False, box_visible = True, meanline_visible = True, side='negative', x0= -3, line_color='blue', opacity=0.6, name='X Error', width=0.9, spanmode='hard'))
fig.add_trace(go.Violin(x=x, y=y_10th_column, points=False, box_visible = True, meanline_visible = True, side='negative', x0 = -2, line_color='red', opacity=0.6, name='Y Error', width=0.9, spanmode='hard'))
fig.add_trace(go.Violin(x=x, y=y_sqrt, points=False, box_visible = True, meanline_visible = True, side='positive', line_color='purple', opacity=0.6, name='Total Error', width=0.9, spanmode='hard'))

fig.update_layout(
    title='Location Prediction Error',
    xaxis=dict(title='Force Range [N]', categoryorder='array', categoryarray=labels),
    # yaxis=dict(title='Location Prediction Error'),
    showlegend=True,  # Show legend to distinguish between columns
    plot_bgcolor='white'
)

fig.update_layout(width=600,height=400,
    yaxis=dict(
        title='Error [mm]',
        range=[-2.5, 2.5]
    )
)

# Display the plot
fig.show()

In [ ]:
# Save the plot as SVG
fig.write_image('exported_figures/others/Violin_test_pressure_map_location_error_results.svg')

### Violin force

In [ ]:
# data_set_locations="sensor_plus_edges"
data_set_locations="sensor_area"

# Define the ranges
ranges = [0.02, 0.5, 1, 1.5, 2]

# Create labels for the ranges
labels = ['[0.02, 0.5)', '[0.5, 1)', '[1, 1.5)', '[1.5, 2]']

if data_set_locations=="sensor_plus_edges":
    filtered_test_dataset = test_error_dataset_force_filtered_cleaned[:] 
elif data_set_locations=="sensor_area":
    filtered_test_dataset = test_error_dataset_force_filtered_cleaned[filtered_indices_inside_edge_sensor]

# Create a new column 'category' based on the ranges
filtered_test_dataset['category'] = pd.cut(filtered_test_dataset.iloc[:, 4], bins=ranges, labels=labels, right=False)

# Specify x values
x = filtered_test_dataset['category']  # Categorical variable created based on the ranges

# Specify y values
F_abs_error=filtered_test_dataset.iloc[:, 14]
F_signed_error=-filtered_test_dataset.iloc[:, 10]-filtered_test_dataset.iloc[:, 4]

# Create violin plots 
fig = go.Figure()

fig.add_trace(go.Violin(x=x, y=F_abs_error, points=False, box_visible = False, meanline_visible = True, side='positive', line_color='#4E99EC', opacity=0.6, name='Force Error', width=1, spanmode='hard'))
fig.add_trace(go.Violin(x=x, y=F_signed_error, points=False, box_visible = False, meanline_visible = True, side='negative', line_color='#3A2D76', opacity=0.6, name='Force Error', width=1, spanmode='soft'))

fig.update_layout(
    title='Force Prediction Error',
    xaxis=dict(title='Force Range [N]', categoryorder='array', categoryarray=labels),
    # yaxis=dict(title='Location Prediction Error'),
    showlegend=True,  # Show legend to distinguish between columns
    plot_bgcolor='white'
)

fig.update_layout(width=950,height=400,
    yaxis=dict(
        title='Error [N]',
        range=[-1,1]
    )
)

# Save the plot as SVG
# fig.write_image('exported_figures/Violin_test_pressuremap_force_error_results.svg')

# Add a delay
time.sleep(1)

# Display the plot
fig.show()

# --- CALCULATE MEDIAN ABSOLUTE ERROR PER RANGE ---
# Grouping by the category and calculating the median of Column 14 (Absolute Force Error)
median_abs_errors = filtered_test_dataset.groupby('category', observed=False).agg({
    filtered_test_dataset.columns[14]: 'median'
}).rename(columns={filtered_test_dataset.columns[14]: 'Median_Absolute_Error'})

print(f"--- Median Absolute Force Errors ({data_set_locations}) ---")
print(median_abs_errors)